<a href="https://colab.research.google.com/github/caihualiang5-svg/protien-Nampt/blob/mutation/2_Structure_and_Microenvironment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2 — WT structure and positional microenvironment

**Input:** the same WT sequence, one or more WT PDB/mmCIF structures (preferred),
optional reference complex/catalytic-residue mapping, and substrate SMILES kept
as metadata. **Output:** `wt_structure_features.csv`, one row per WT position.

The coarse stage models WT only. Smoke mode uses Biopython ShrakeRupley and no
heavy installation. Production uses the FreeSASA command-line program plus
DSSP; ColabFold is installed only when `RUN_COLABFOLD=True`.


In [1]:

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
# 所有用户配置都显式赋值，禁止复用旧运行时状态。
TEST_MODE = False  # 正式运行保持 False；50 aa 烟雾测试时临时改为 True。
INPUT_MODE = "drive"  # "drive" 或 "upload"
INSTALL_DEPENDENCIES = True
if INPUT_MODE not in {"drive", "upload"}:
    raise ValueError("INPUT_MODE must be 'drive' or 'upload'")

SMOKE_WT_SEQUENCE = "ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL"
# 正式运行：把三引号中的占位文字替换为与 Notebook 1 完全相同的完整序列。
USER_WT_SEQUENCE = """
MQPNIILLTDSYKLSHYKQYPAGTSQIYSYFESRGGEFEGVTFFGLQYLLKEYLEGQVVT
QEKIDRADKIYAAHFGTEKLFNKAGWEYILHTHNGHLPIRIKAVAEGTVIPTHNVMLTIE
NTDPNCFWLTNFLETLLLQLWYPCTVATISREVKTLITKYLEETGDPSTIDFKLHDFGFR
GVSSVQSAGIGGAAHLVNFMGTDTVAALTFIQEYYAPFPVGEGLGMGFPMFGFSIPAAEH
STITSWGRDNETDAYQNMLQQYPEGLVAVVSDSYDIYNACEKIWGEVLKDNILQRNGTLV
VRPDSGEPKDVVLKCTQILGEKIGYSINEKGYKVLNPKIRIIQGDGVNYESIGEILEHLK
KHGWSADNVAFGMGGALLQKVHRDTQKFAFKCSCATVNGEDRDVYKDPATDHGKKSKRGR
LKLVKENEMYITKAINEDGEDILQTVFENGKILREIDFQGVKENNLK
"""
FASTA_FILENAME = ""  # 也可填写 inputs/ 中的 FASTA 文件名，优先于上方粘贴序列。
PROTEIN_ID = "FAKE50" if TEST_MODE else "NAMPT_WT"
WT_SEQUENCE = USER_WT_SEQUENCE
if TEST_MODE:
    WT_SEQUENCE = SMOKE_WT_SEQUENCE

NAM_SMILES = "NC(=O)c1ccncc1"
PRPP_SMILES = "O=P(O)(O)OCC1OC(OP(=O)(O)O)C(O)C1O"

# 不再使用烟雾测试的假位点 {10, 20}。正式模式默认自动映射人源 NAMPT
# 的 D219/H247/D313 到你的序列；如已实验确认，可关闭自动映射并填写手工编号。
MANUAL_CATALYTIC_RESIDUES = set()
AUTO_MAP_CATALYTIC_RESIDUES = True
CATALYTIC_RESIDUES = set()
if MANUAL_CATALYTIC_RESIDUES:
    CATALYTIC_RESIDUES = set(MANUAL_CATALYTIC_RESIDUES)
HUMAN_NAMPT_FASTA_URL = "https://rest.uniprot.org/uniprotkb/P43490.fasta"
REFERENCE_ACTIVE_SITE_DEFINITIONS = {
    219: {"expected_aa": "D", "role": "NAM substrate-specificity anchor"},
    247: {"expected_aa": "H", "role": "phosphorylated catalytic histidine"},
    313: {"expected_aa": "D", "role": "transition-state catalytic-site anchor"},
}
MIN_REFERENCE_IDENTITY = 0.25
MIN_REFERENCE_COVERAGE = 0.70

TARGET_CHAIN_ID = "A"
SMOKE_SASA_BACKEND = "biopython_smoke"
SMOKE_OUTPUT_DIR = Path("/content")

# 已有 WT 二聚体 PDB/mmCIF 时会优先使用；没有时自动运行 ColabFold。
RUN_COLABFOLD = True
COLABFOLD_NUM_MODELS = 3
COLABFOLD_NUM_RECYCLES = 3
NEED_FREESASA = True
NEED_DSSP = True

# Drive 模式：文件名相对于 MyDrive/nampt_zero_shot/inputs/；留空时自动找 PDB/mmCIF。
WT_STRUCTURE_FILENAMES = []
AUTO_DOWNLOAD_REFERENCE_COMPLEX = True
REFERENCE_COMPLEX_PDB_ID = "3DKL"
REFERENCE_COMPLEX_FILENAME = "3DKL.pdb"
REFERENCE_COMPLEX_URL = "https://files.rcsb.org/download/3DKL.pdb"
REFERENCE_CHAIN_ID = "A"
DRIVE_DIR = Path("/content/drive/MyDrive/nampt_zero_shot")
INPUT_DIR = DRIVE_DIR / "inputs"
MODULE2_DIR = DRIVE_DIR / "module2"
STRUCTURE_LOG_DIR = MODULE2_DIR / "logs"
LOCAL_OUTPUT_DIR = Path("/content/nampt_zero_shot")

print("TEST_MODE:", TEST_MODE)
print("RUN_COLABFOLD:", RUN_COLABFOLD)
print("WT sequence source:", FASTA_FILENAME or "USER_WT_SEQUENCE")


TEST_MODE: False
RUN_COLABFOLD: True
WT sequence source: USER_WT_SEQUENCE


In [2]:

DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    for directory in [DRIVE_DIR, INPUT_DIR, MODULE2_DIR, LOCAL_OUTPUT_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
else:
    print("Drive mount is available only in Colab.")


Mounted at /content/drive


In [3]:

UPLOADED_INPUTS = {}
UPLOADED_PATHS = {}
if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("INPUT_MODE='upload' requires a Colab runtime")
    from google.colab import files
    UPLOADED_INPUTS = files.upload()
    UPLOADED_PATHS = {
        name: Path("/content") / name for name in UPLOADED_INPUTS
    }
    print("Uploaded:", sorted(UPLOADED_PATHS))
else:
    print(f"INPUT_MODE='drive': reading inputs from {INPUT_DIR}; no upload dialog.")


INPUT_MODE='drive': reading inputs from /content/drive/MyDrive/nampt_zero_shot/inputs; no upload dialog.


In [4]:

import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path


def module2_dependency_plan(
    test_mode, run_colabfold, need_freesasa=True, need_dssp=True,
    has_structure_input=False,
):
    """Return only mode-relevant dependencies; never downgrade Colab core packages."""
    pip_requirements = []
    apt_requirements = []
    if importlib.util.find_spec("Bio") is None:
        pip_requirements.append("biopython>=1.85")
    if not test_mode and need_freesasa and shutil.which("freesasa") is None:
        apt_requirements.append("freesasa")
    if not test_mode and need_dssp and shutil.which("mkdssp") is None:
        apt_requirements.append("dssp")
    if not test_mode and run_colabfold and not has_structure_input:
        if importlib.util.find_spec("colabfold") is None or importlib.util.find_spec("alphafold") is None:
            pip_requirements.append("colabfold[alphafold,openmm]>=1.6.2")
        if importlib.util.find_spec("jax") is None:
            pip_requirements.append("jax[cuda12]")
        if importlib.util.find_spec("openmm") is None:
            pip_requirements.append("openmm[cuda12]")
    return {"pip": pip_requirements, "apt": apt_requirements}


def _structure_input_is_available():
    suffixes = {".pdb", ".cif", ".mmcif"}
    reference_name = Path(REFERENCE_COMPLEX_FILENAME).name

    def is_wt_structure(path):
        path = Path(path)
        return (
            path.suffix.lower() in suffixes
            and (not reference_name or path.name != reference_name)
        )

    if any(
        is_wt_structure(name)
        for name in WT_STRUCTURE_FILENAMES
    ):
        return True
    if INPUT_MODE == "upload":
        return any(
            is_wt_structure(path)
            for path in UPLOADED_PATHS.values()
        )
    return INPUT_DIR.exists() and any(
        path.is_file() and is_wt_structure(path)
        for path in INPUT_DIR.iterdir()
    )


DEPENDENCY_PLAN = module2_dependency_plan(
    TEST_MODE,
    RUN_COLABFOLD,
    NEED_FREESASA,
    NEED_DSSP,
    has_structure_input=_structure_input_is_available(),
)
print("Notebook 2 dependency plan:", DEPENDENCY_PLAN)
if INSTALL_DEPENDENCIES:
    if not IN_COLAB:
        print("Not running in Colab; installation skipped.")
    else:
        if DEPENDENCY_PLAN["pip"]:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"]
                + DEPENDENCY_PLAN["pip"]
            )
        if DEPENDENCY_PLAN["apt"]:
            import os
            apt_environment = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
            subprocess.check_call(
                ["apt-get", "update", "-qq"], env=apt_environment
            )
            subprocess.check_call(
                ["apt-get", "install", "-y", "-qq"] + DEPENDENCY_PLAN["apt"],
                env=apt_environment,
            )
else:
    print("Dependency installation disabled; plan only.")

if IN_COLAB and INSTALL_DEPENDENCIES and not TEST_MODE:
    missing_commands = []
    if NEED_FREESASA and shutil.which("freesasa") is None:
        missing_commands.append("freesasa")
    if NEED_DSSP and not (shutil.which("mkdssp") or shutil.which("dssp")):
        missing_commands.append("mkdssp/dssp")
    if missing_commands:
        raise RuntimeError(
            "System dependency installation failed: " + ", ".join(missing_commands)
        )
    print("FreeSASA:", shutil.which("freesasa"))
    print("DSSP:", shutil.which("mkdssp") or shutil.which("dssp"))


Notebook 2 dependency plan: {'pip': ['biopython>=1.85', 'colabfold[alphafold,openmm]>=1.6.2', 'openmm[cuda12]'], 'apt': ['freesasa', 'dssp']}
FreeSASA: /usr/bin/freesasa
DSSP: /usr/bin/mkdssp


In [5]:
import hashlib
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.Align import PairwiseAligner
from Bio.PDB import (
    DSSP, MMCIFParser, NeighborSearch, PDBIO, PDBParser, Select, Superimposer
)
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.vectors import calc_dihedral

AA20 = "ACDEFGHIKLMNPQRSTVWY"
AA3 = {
    "A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU",
    "F": "PHE", "G": "GLY", "H": "HIS", "I": "ILE",
    "K": "LYS", "L": "LEU", "M": "MET", "N": "ASN",
    "P": "PRO", "Q": "GLN", "R": "ARG", "S": "SER",
    "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR",
}
AA1_FROM_3 = {three: one for one, three in AA3.items()}
MAX_ASA = {
    "ALA": 129.0, "ARG": 274.0, "ASN": 195.0, "ASP": 193.0,
    "CYS": 167.0, "GLN": 225.0, "GLU": 223.0, "GLY": 104.0,
    "HIS": 224.0, "ILE": 197.0, "LEU": 201.0, "LYS": 236.0,
    "MET": 224.0, "PHE": 240.0, "PRO": 159.0, "SER": 155.0,
    "THR": 172.0, "TRP": 285.0, "TYR": 263.0, "VAL": 174.0,
}
BACKBONE_ATOMS = {"N", "CA", "C", "O", "OXT"}


def validate_wt_sequence(sequence: str) -> str:
    cleaned = "".join(sequence.split()).upper()
    invalid = sorted(set(cleaned) - set(AA20))
    if not cleaned or invalid:
        raise ValueError(f"Invalid WT sequence residues: {invalid}")
    return cleaned


def map_reference_active_site_residues(
    reference_sequence: str,
    target_sequence: str,
    site_definitions: dict[int, dict[str, str]],
    minimum_identity: float = 0.25,
    minimum_reference_coverage: float = 0.70,
) -> tuple[pd.DataFrame, dict]:
    """Map curated reference positions without copying reference numbering blindly."""
    reference_sequence = validate_wt_sequence(reference_sequence)
    target_sequence = validate_wt_sequence(target_sequence)
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2.0
    aligner.mismatch_score = -1.0
    aligner.open_gap_score = -8.0
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(reference_sequence, target_sequence)[0]
    position_map = {}
    for reference_block, target_block in zip(
        alignment.aligned[0], alignment.aligned[1]
    ):
        reference_start, reference_end = map(int, reference_block)
        target_start, target_end = map(int, target_block)
        block_length = min(
            reference_end - reference_start,
            target_end - target_start,
        )
        for offset in range(block_length):
            position_map[reference_start + offset + 1] = target_start + offset + 1
    identical_pairs = sum(
        reference_sequence[reference_position - 1]
        == target_sequence[target_position - 1]
        for reference_position, target_position in position_map.items()
    )
    aligned_pairs = len(position_map)
    identity = identical_pairs / aligned_pairs if aligned_pairs else 0.0
    reference_coverage = aligned_pairs / len(reference_sequence)
    target_coverage = aligned_pairs / len(target_sequence)
    mapping_accepted = (
        identity >= minimum_identity
        and reference_coverage >= minimum_reference_coverage
    )
    rows = []
    for reference_position, definition in sorted(site_definitions.items()):
        expected_aa = str(definition["expected_aa"]).upper()
        reference_aa = reference_sequence[reference_position - 1]
        target_position = position_map.get(reference_position)
        target_aa = (
            target_sequence[target_position - 1]
            if target_position is not None else None
        )
        if reference_aa != expected_aa:
            status = "reference_residue_mismatch"
        elif target_position is None:
            status = "alignment_gap"
        elif target_aa != expected_aa:
            status = "residue_mismatch"
        elif not mapping_accepted:
            status = "overall_alignment_rejected"
        else:
            status = "accepted"
        rows.append({
            "reference_position": int(reference_position),
            "reference_aa": reference_aa,
            "target_position": target_position,
            "target_aa": target_aa,
            "role": definition["role"],
            "residue_conserved": status == "accepted",
            "mapping_status": status,
            "overall_mapping_accepted": mapping_accepted,
        })
    frame = pd.DataFrame(rows)
    summary = {
        "aligned_pairs": aligned_pairs,
        "identical_pairs": identical_pairs,
        "sequence_identity": identity,
        "reference_coverage": reference_coverage,
        "target_coverage": target_coverage,
        "minimum_identity": minimum_identity,
        "minimum_reference_coverage": minimum_reference_coverage,
        "mapping_accepted": mapping_accepted,
    }
    return frame, summary


def accepted_mapped_positions(mapping_frame: pd.DataFrame) -> set[int]:
    if mapping_frame.empty:
        return set()
    accepted = mapping_frame.loc[
        mapping_frame["mapping_status"].eq("accepted"), "target_position"
    ].dropna()
    return set(accepted.astype(int))


def write_synthetic_pdb(sequence: str, path: str | Path) -> Path:
    sequence = validate_wt_sequence(sequence)
    serial = 1
    lines = []
    offsets = {
        "N": (-1.20, 0.00, 0.00),
        "CA": (0.00, 0.00, 0.00),
        "C": (1.30, 0.00, 0.00),
        "O": (2.10, 0.55, 0.00),
        "CB": (0.00, 1.50, 0.70),
    }
    for position, aa in enumerate(sequence, start=1):
        base_x = 3.80 * (position - 1)
        atom_names = ["N", "CA", "C", "O"] + (
            [] if aa == "G" else ["CB"]
        )
        for atom_name in atom_names:
            dx, dy, dz = offsets[atom_name]
            element = atom_name[0]
            lines.append(
                f"ATOM  {serial:5d} {atom_name:^4s} {AA3[aa]:>3s} "
                f"A{position:4d}    {base_x + dx:8.3f}{dy:8.3f}{dz:8.3f}"
                f"  1.00 20.00          {element:>2s}\n"
            )
            serial += 1
    lines.append("END\n")
    output = Path(path)
    output.write_text("".join(lines), encoding="ascii")
    return output


def load_structure(path: str | Path, structure_id: str = "structure"):
    """Load PDB or mmCIF while keeping one parser contract downstream."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".cif", ".mmcif"}:
        parser = MMCIFParser(QUIET=True)
    elif suffix in {".pdb", ".ent"}:
        parser = PDBParser(QUIET=True)
    else:
        raise ValueError(f"Unsupported structure format: {path.suffix}")
    return parser.get_structure(structure_id, str(path))


def parse_structure_residue_map(
    pdb_path: str | Path, chain_id: str = "A"
) -> dict[int, object]:
    structure = load_structure(pdb_path, "wt")
    model = next(structure.get_models())
    if chain_id not in model:
        raise ValueError(f"Chain {chain_id!r} not found")
    residues = [
        residue for residue in model[chain_id]
        if is_aa(residue, standard=True)
    ]
    return {
        position: residue
        for position, residue in enumerate(residues, start=1)
    }


def residue_identifier(residue):
    number = int(residue.id[1])
    insertion_code = str(residue.id[2]).strip()
    return number, insertion_code, f"{number}:{insertion_code}"


def parse_freesasa_residue_identifier(value):
    match = re.fullmatch(
        r"\s*(-?\d+)\s*([A-Za-z]?)\s*", str(value)
    )
    if match is None:
        raise ValueError(f"Unrecognized FreeSASA residue identifier: {value!r}")
    number = int(match.group(1))
    insertion_code = match.group(2)
    return number, insertion_code, f"{number}:{insertion_code}"


def align_structure_residues_to_wt(
    sequence: str, pdb_path: str | Path, chain_id: str = "A"
) -> tuple[dict[int, object], dict[int, str]]:
    # Map resolved chain residues to WT positions by global alignment.
    observed_map = parse_structure_residue_map(pdb_path, chain_id)
    observed_residues = list(observed_map.values())
    observed_sequence = "".join(
        AA1_FROM_3.get(residue.get_resname().upper(), "X")
        for residue in observed_residues
    )
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2.0
    aligner.mismatch_score = -1.0
    aligner.open_gap_score = -5.0
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(sequence, observed_sequence)[0]
    mapped = {}
    statuses = {
        position: "missing_in_structure"
        for position in range(1, len(sequence) + 1)
    }
    target_blocks, query_blocks = alignment.aligned
    for (wt_start, wt_end), (obs_start, obs_end) in zip(
        target_blocks, query_blocks
    ):
        block_length = min(wt_end - wt_start, obs_end - obs_start)
        for offset in range(block_length):
            wt_index = int(wt_start + offset)
            observed_index = int(obs_start + offset)
            if sequence[wt_index] != observed_sequence[observed_index]:
                statuses[wt_index + 1] = "identity_mismatch"
                continue
            mapped[wt_index + 1] = observed_residues[observed_index]
            statuses[wt_index + 1] = "aligned_identity"
    if len(mapped) / len(sequence) < 0.5:
        raise ValueError(
            "Fewer than 50% of WT residues map identically to the selected chain"
        )
    return mapped, statuses


def _relative_sasa_percent(value: float) -> float:
    value = float(value)
    return value * 100.0 if value <= 1.5 else value



def _parse_freesasa_rsa(text: str, chain_id: str) -> pd.DataFrame:
    """Parse residue-level FreeSASA RSA output without Python bindings."""
    rows = []
    for line in text.splitlines():
        if not line.startswith("RES"):
            continue
        fields = line.split()
        if len(fields) < 8 or fields[2] != chain_id:
            continue
        residue_name = fields[1].upper()
        number, insertion_code, residue_key = parse_freesasa_residue_identifier(
            fields[3]
        )
        total_sasa = float(fields[4])
        relative_sasa = (
            100.0 * total_sasa / MAX_ASA[residue_name]
            if fields[5].upper() == "N/A"
            else float(fields[5])
        )
        rows.append({
            "pdb_residue_key": residue_key,
            "pdb_residue_number": number,
            "pdb_insertion_code": insertion_code,
            "relative_sasa": relative_sasa,
            "total_sasa": total_sasa,
            "sidechain_sasa": float(fields[6]),
            "sasa_backend": "freesasa_cli",
        })
    return pd.DataFrame(rows)


def _calculate_freesasa_cli_rows(
    pdb_path: str | Path,
    chain_id: str,
    artifact_dir: str | Path | None = None,
    artifact_prefix: str | None = None,
) -> pd.DataFrame:
    import shutil
    import subprocess

    if shutil.which("freesasa") is None:
        raise RuntimeError(
            "FreeSASA CLI is missing. Rerun config + install with "
            "TEST_MODE=False and NEED_FREESASA=True."
        )
    pdb_path = Path(pdb_path)
    command = ["freesasa", "--format=rsa"]
    if pdb_path.suffix.lower() in {".cif", ".mmcif"}:
        command.insert(1, "--cif")
    command.append(str(pdb_path))
    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
    )
    if artifact_dir is not None:
        artifact_dir = Path(artifact_dir)
        artifact_dir.mkdir(parents=True, exist_ok=True)
        safe_prefix = re.sub(
            r"[^A-Za-z0-9_.-]+", "_", artifact_prefix or pdb_path.stem
        )
        (artifact_dir / f"{safe_prefix}_freesasa.rsa").write_text(
            completed.stdout or "", encoding="utf-8"
        )
        (artifact_dir / f"{safe_prefix}_freesasa.stderr.log").write_text(
            completed.stderr or "", encoding="utf-8"
        )
        (artifact_dir / f"{safe_prefix}_freesasa.command.txt").write_text(
            " ".join(command) + "\n", encoding="utf-8"
        )
    if completed.returncode != 0:
        stderr_tail = (completed.stderr or "")[-4000:]
        print("===== FreeSASA stderr tail =====\n" + stderr_tail)
        raise RuntimeError(
            f"FreeSASA CLI failed (exit {completed.returncode}).\n{stderr_tail}"
        )
    return _parse_freesasa_rsa(completed.stdout, chain_id)


def _calculate_biopython_smoke_rows(
    pdb_path: str | Path, chain_id: str
) -> pd.DataFrame:
    structure = load_structure(pdb_path, "smoke")
    model = next(structure.get_models())
    if chain_id not in model:
        raise ValueError(f"Chain {chain_id!r} not found")
    ShrakeRupley().compute(model, level="A")
    rows = []
    for residue in model[chain_id]:
        if not is_aa(residue, standard=True):
            continue
        atoms = [atom for atom in residue.get_atoms() if atom.element != "H"]
        total = float(sum(float(atom.sasa) for atom in atoms))
        sidechain = float(
            sum(
                float(atom.sasa) for atom in atoms
                if atom.get_name().strip() not in BACKBONE_ATOMS
            )
        )
        max_asa = MAX_ASA[residue.get_resname().upper()]
        number, insertion_code, residue_key = residue_identifier(residue)
        rows.append({
            "pdb_residue_key": residue_key,
            "pdb_residue_number": number,
            "pdb_insertion_code": insertion_code,
            "relative_sasa": 100.0 * total / max_asa,
            "total_sasa": total,
            "sidechain_sasa": sidechain,
            "sasa_backend": "biopython_smoke",
        })
    return pd.DataFrame(rows)


def calculate_residue_sasa(
    pdb_path: str | Path,
    chain_id: str = "A",
    backend: str = "freesasa_cli",
    artifact_dir: str | Path | None = None,
    artifact_prefix: str | None = None,
) -> pd.DataFrame:
    if backend == "freesasa_cli":
        rows = _calculate_freesasa_cli_rows(
            pdb_path, chain_id, artifact_dir, artifact_prefix
        )
    elif backend == "freesasa":
        rows = _calculate_freesasa_cli_rows(
            pdb_path, chain_id, artifact_dir, artifact_prefix
        )
    elif backend == "biopython_smoke":
        rows = _calculate_biopython_smoke_rows(pdb_path, chain_id)
    else:
        raise ValueError(f"Unsupported SASA backend: {backend}")
    if rows.empty:
        raise ValueError(f"SASA backend returned no residues for chain {chain_id!r}")
    numeric = rows[["relative_sasa", "total_sasa", "sidechain_sasa"]]
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError("SASA output contains non-finite values")
    if (numeric.to_numpy(dtype=float) < 0).any():
        raise ValueError("SASA output contains negative values")
    return rows


def minimum_heavy_atom_distance(
    residue, reference_atoms: list[object]
) -> float:
    residue_atoms = [
        atom for atom in residue.get_atoms() if atom.element != "H"
    ]
    if not residue_atoms or not reference_atoms:
        return float("nan")
    return min(
        float(atom_a - atom_b)
        for atom_a in residue_atoms
        for atom_b in reference_atoms
    )


def extract_named_ligand_atoms(
    pdb_path: str | Path, residue_names: set[str]
) -> list[object]:
    normalized = {name.strip().upper() for name in residue_names}
    structure = load_structure(pdb_path, "ligands")
    model = next(structure.get_models())
    return [
        atom for chain in model for residue in chain
        if residue.get_resname().strip().upper() in normalized
        for atom in residue.get_atoms() if atom.element != "H"
    ]


def transfer_reference_ligands(
    target_pdb_path: str | Path,
    reference_pdb_path: str | Path,
    target_chain_id: str = "A",
    reference_chain_id: str = "A",
    ligand_resnames: dict[str, set[str]] | None = None,
    minimum_ca_pairs: int = 20,
    minimum_coverage: float = 0.50,
    maximum_rmsd: float = 3.0,
) -> dict:
    import copy

    ligand_resnames = ligand_resnames or {
        "NAM": {"NAM"}, "PRPP": {"PRP", "PRPP"}
    }

    def failed(reason, count=0, coverage=0.0, rmsd=float("nan")):
        return {
            "accepted": False,
            "aligned_ca_count": int(count),
            "aligned_coverage": float(coverage),
            "rmsd_angstrom": float(rmsd),
            "nam_atoms": [],
            "prpp_atoms": [],
            "failure_reason": reason,
        }

    target_structure = load_structure(target_pdb_path, "target")
    reference_structure = load_structure(reference_pdb_path, "reference")
    target_model = next(target_structure.get_models())
    reference_model = next(reference_structure.get_models())
    if target_chain_id not in target_model:
        return failed(f"target chain {target_chain_id!r} not found")
    if reference_chain_id not in reference_model:
        return failed(f"reference chain {reference_chain_id!r} not found")
    target_residues = [
        residue for residue in target_model[target_chain_id]
        if is_aa(residue, standard=True) and "CA" in residue
    ]
    reference_residues = [
        residue for residue in reference_model[reference_chain_id]
        if is_aa(residue, standard=True) and "CA" in residue
    ]
    target_sequence = "".join(
        AA1_FROM_3.get(residue.get_resname().upper(), "X")
        for residue in target_residues
    )
    reference_sequence = "".join(
        AA1_FROM_3.get(residue.get_resname().upper(), "X")
        for residue in reference_residues
    )
    if "X" in target_sequence or "X" in reference_sequence:
        return failed("nonstandard residue prevented sequence mapping")
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2.0
    aligner.mismatch_score = -1.0
    aligner.open_gap_score = -5.0
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(reference_sequence, target_sequence)[0]
    reference_ca = []
    target_ca = []
    for reference_block, target_block in zip(
        alignment.aligned[0], alignment.aligned[1]
    ):
        reference_start, reference_end = map(int, reference_block)
        target_start, target_end = map(int, target_block)
        block_length = min(
            reference_end - reference_start,
            target_end - target_start,
        )
        for offset in range(block_length):
            reference_ca.append(
                reference_residues[reference_start + offset]["CA"]
            )
            target_ca.append(target_residues[target_start + offset]["CA"])
    count = len(reference_ca)
    coverage = count / max(1, len(target_residues))
    if count < minimum_ca_pairs:
        return failed("fewer than the required aligned C-alpha pairs", count, coverage)
    if coverage < minimum_coverage:
        return failed("aligned coverage below threshold", count, coverage)
    superimposer = Superimposer()
    superimposer.set_atoms(target_ca, reference_ca)
    rmsd = float(superimposer.rms)
    if rmsd > maximum_rmsd:
        return failed("C-alpha RMSD above threshold", count, coverage, rmsd)
    copied = {"NAM": [], "PRPP": []}
    normalized_names = {
        substrate: {name.upper() for name in names}
        for substrate, names in ligand_resnames.items()
    }
    for chain in reference_model:
        for residue in chain:
            residue_name = residue.get_resname().strip().upper()
            for substrate in ["NAM", "PRPP"]:
                if residue_name in normalized_names.get(substrate, set()):
                    copied[substrate].extend(
                        copy.deepcopy(atom)
                        for atom in residue.get_atoms()
                        if atom.element != "H"
                    )
    ligand_atoms = copied["NAM"] + copied["PRPP"]
    if not ligand_atoms:
        return failed("no recognized NAM or PRPP atoms", count, coverage, rmsd)
    superimposer.apply(ligand_atoms)
    return {
        "accepted": True,
        "aligned_ca_count": count,
        "aligned_coverage": coverage,
        "rmsd_angstrom": rmsd,
        "nam_atoms": copied["NAM"],
        "prpp_atoms": copied["PRPP"],
        "failure_reason": "",
    }


def calculate_backbone_angles(
    residue_map: dict[int, object]
) -> dict[int, tuple[float, float]]:
    import warnings

    angles = {}
    for position, residue in residue_map.items():
        phi = float("nan")
        psi = float("nan")
        if position - 1 in residue_map:
            previous = residue_map[position - 1]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                phi = math.degrees(
                    float(calc_dihedral(
                        previous["C"].get_vector(),
                        residue["N"].get_vector(),
                        residue["CA"].get_vector(),
                        residue["C"].get_vector(),
                    ))
                )
        if position + 1 in residue_map:
            following = residue_map[position + 1]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                psi = math.degrees(
                    float(calc_dihedral(
                        residue["N"].get_vector(),
                        residue["CA"].get_vector(),
                        residue["C"].get_vector(),
                        following["N"].get_vector(),
                    ))
                )
        angles[position] = (phi, psi)
    return angles


class _OnlySelectedChain(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id

    def accept_chain(self, chain):
        return int(chain.id == self.chain_id)


def calculate_interface_features(
    pdb_path: str | Path,
    chain_id: str = "A",
    sasa_backend: str = "freesasa_cli",
    artifact_dir: str | Path | None = None,
    artifact_prefix: str | None = None,
) -> pd.DataFrame:
    import tempfile

    structure = load_structure(pdb_path, "complex")
    model = next(structure.get_models())
    if chain_id not in model:
        raise ValueError(f"Chain {chain_id!r} not found")
    target_residues = [
        residue for residue in model[chain_id]
        if is_aa(residue, standard=True)
    ]
    residue_identity = pd.DataFrame([
        {
            "pdb_residue_key": residue_identifier(residue)[2],
            "pdb_residue_number": residue_identifier(residue)[0],
            "pdb_insertion_code": residue_identifier(residue)[1],
        }
        for residue in target_residues
    ])
    partner_atoms = [
        atom for chain in model if chain.id != chain_id
        for residue in chain if is_aa(residue, standard=True)
        for atom in residue if atom.element != "H"
    ]
    if not partner_atoms:
        result = residue_identity.copy()
        result["interface_delta_sasa"] = 0.0
        result["partner_heavy_atom_contact"] = False
        result["interface_partner_present"] = False
        return result
    partner_search = NeighborSearch(partner_atoms)
    complex_sasa = calculate_residue_sasa(
        pdb_path, chain_id, backend=sasa_backend,
        artifact_dir=artifact_dir,
        artifact_prefix=(
            f"{artifact_prefix}_interface_complex"
            if artifact_prefix else "interface_complex"
        ),
    )[["pdb_residue_key", "total_sasa"]].rename(
        columns={"total_sasa": "complex_total_sasa"}
    )
    temporary = tempfile.NamedTemporaryFile(
        suffix=".pdb", delete=False
    )
    temporary.close()
    isolated_path = Path(temporary.name)
    try:
        writer = PDBIO()
        writer.set_structure(structure)
        writer.save(str(isolated_path), _OnlySelectedChain(chain_id))
        isolated_sasa = calculate_residue_sasa(
            isolated_path, chain_id, backend=sasa_backend,
            artifact_dir=artifact_dir,
            artifact_prefix=(
                f"{artifact_prefix}_interface_isolated"
                if artifact_prefix else "interface_isolated"
            ),
        )[["pdb_residue_key", "total_sasa"]].rename(
            columns={"total_sasa": "isolated_total_sasa"}
        )
    finally:
        isolated_path.unlink(missing_ok=True)
    areas = complex_sasa.merge(
        isolated_sasa, on="pdb_residue_key", validate="one_to_one"
    )
    areas = residue_identity.merge(
        areas, on="pdb_residue_key", validate="one_to_one"
    )
    areas["interface_delta_sasa"] = (
        areas["isolated_total_sasa"] - areas["complex_total_sasa"]
    ).clip(lower=0.0)
    contact_by_key = {}
    for residue in target_residues:
        contact_by_key[residue_identifier(residue)[2]] = any(
            partner_search.search(atom.coord, 5.0, level="A")
            for atom in residue if atom.element != "H"
        )
    areas["partner_heavy_atom_contact"] = areas[
        "pdb_residue_key"
    ].map(contact_by_key).fillna(False)
    areas["interface_partner_present"] = True
    return areas[[
        "pdb_residue_key", "pdb_residue_number",
        "pdb_insertion_code", "interface_delta_sasa",
        "partner_heavy_atom_contact", "interface_partner_present",
    ]]


def prepare_dssp_input(
    pdb_path: str | Path,
    compatibility_path: str | Path,
) -> tuple[Path, bool]:
    """Create a DSSP-compatible PDB copy without modifying the source model."""
    pdb_path = Path(pdb_path)
    compatibility_path = Path(compatibility_path)
    if pdb_path.suffix.lower() != ".pdb":
        return pdb_path, False

    original = pdb_path.read_text(encoding="utf-8", errors="replace")
    lines = original.splitlines(keepends=True)
    starts_with_header = bool(lines) and lines[0].startswith("HEADER")
    has_cryst1 = any(line.startswith("CRYST1") for line in lines)
    if starts_with_header and has_cryst1:
        return pdb_path, False

    prepared_lines = []
    if not starts_with_header:
        prepared_lines.append(
            "HEADER    COLABFOLD PREDICTED STRUCTURE             "
            "01-JAN-00   CF01\n"
        )
    else:
        prepared_lines.append(lines.pop(0))
    if not has_cryst1:
        prepared_lines.append(
            "CRYST1  999.000  999.000  999.000  90.00  90.00  90.00 "
            "P 1           1\n"
        )
    prepared_lines.extend(lines)
    compatibility_path.parent.mkdir(parents=True, exist_ok=True)
    compatibility_path.write_text(
        "".join(prepared_lines),
        encoding="utf-8",
    )
    return compatibility_path, True


def calculate_dssp_features(
    pdb_path: str | Path,
    chain_id: str = "A",
    artifact_dir: str | Path | None = None,
    artifact_prefix: str | None = None,
) -> pd.DataFrame:
    import os
    import shutil
    import tempfile
    import traceback

    pdb_path = Path(pdb_path)
    residue_map = parse_structure_residue_map(pdb_path, chain_id)
    residue_identity = [
        {
            "pdb_residue_key": residue_identifier(residue)[2],
            "pdb_residue_number": residue_identifier(residue)[0],
            "pdb_insertion_code": residue_identifier(residue)[1],
        }
        for residue in residue_map.values()
    ]
    log_path = None
    raw_path = None
    compatibility_path = None
    if artifact_dir is not None:
        artifact_dir = Path(artifact_dir)
        artifact_dir.mkdir(parents=True, exist_ok=True)
        safe_prefix = re.sub(
            r"[^A-Za-z0-9_.-]+", "_", artifact_prefix or pdb_path.stem
        )
        log_path = artifact_dir / f"{safe_prefix}_dssp.log"
        raw_path = artifact_dir / f"{safe_prefix}_dssp_raw.csv"
        compatibility_path = artifact_dir / f"{safe_prefix}_dssp_input.pdb"
    executable = shutil.which("mkdssp") or shutil.which("dssp")
    if executable is None:
        result = pd.DataFrame(residue_identity)
        result["secondary_structure_dssp"] = "unknown"
        result["phi_dssp"] = np.nan
        result["psi_dssp"] = np.nan
        result["dssp_status"] = "executable_unavailable"
        if log_path is not None:
            log_path.write_text(
                f"status=executable_unavailable\ninput={pdb_path}\n",
                encoding="utf-8",
            )
        if raw_path is not None:
            result.to_csv(raw_path, index=False)
        return result
    temporary_path = None
    if compatibility_path is None:
        descriptor, temporary_name = tempfile.mkstemp(
            prefix="nampt_dssp_", suffix=".pdb"
        )
        os.close(descriptor)
        temporary_path = Path(temporary_name)
        compatibility_path = temporary_path
    dssp_input_path, dssp_input_modified = prepare_dssp_input(
        pdb_path, compatibility_path
    )
    if temporary_path is not None and not dssp_input_modified:
        temporary_path.unlink(missing_ok=True)
        temporary_path = None

    structure = load_structure(pdb_path, "dssp")
    model = next(structure.get_models())
    try:
        dssp = DSSP(model, str(dssp_input_path), dssp=executable)
    except Exception as exc:
        details = traceback.format_exc()
        print("===== DSSP failure =====\n" + details)
        if log_path is not None:
            log_path.write_text(
                f"input={pdb_path}\n"
                f"dssp_input={dssp_input_path}\n"
                f"compatibility_copy={dssp_input_modified}\n"
                + details,
                encoding="utf-8",
            )
        result = pd.DataFrame(residue_identity)
        result["secondary_structure_dssp"] = "unknown"
        result["phi_dssp"] = np.nan
        result["psi_dssp"] = np.nan
        result["dssp_status"] = (
            f"failed:{type(exc).__name__}:{str(exc)[:500]}"
        )
        if raw_path is not None:
            result.to_csv(raw_path, index=False)
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
        return result
    if temporary_path is not None:
        temporary_path.unlink(missing_ok=True)
    rows = []
    for residue in residue_map.values():
        key = (chain_id, residue.id)
        number, insertion_code, residue_key = residue_identifier(residue)
        if key not in dssp:
            rows.append({
                "pdb_residue_key": residue_key,
                "pdb_residue_number": number,
                "pdb_insertion_code": insertion_code,
                "secondary_structure_dssp": "unknown",
                "phi_dssp": np.nan,
                "psi_dssp": np.nan,
                "dssp_status": "residue_unmapped",
            })
            continue
        record = dssp[key]
        rows.append({
            "pdb_residue_key": residue_key,
            "pdb_residue_number": number,
            "pdb_insertion_code": insertion_code,
            "secondary_structure_dssp": record[2] if record[2] != "-" else "coil",
            "phi_dssp": float(record[4]),
            "psi_dssp": float(record[5]),
            "dssp_status": "success",
        })
    result = pd.DataFrame(rows)
    if raw_path is not None:
        result.to_csv(raw_path, index=False)
    if log_path is not None:
        log_path.write_text(
            f"status=success\ninput={pdb_path}\nexecutable={executable}\n"
            f"dssp_input={dssp_input_path}\n"
            f"compatibility_copy={dssp_input_modified}\n"
            f"residue_rows={len(result)}\n",
            encoding="utf-8",
        )
    return result


def assign_structure_bin(distance: float) -> str:
    if pd.isna(distance):
        return "unknown"
    if distance <= 4.5:
        return "direct_contact"
    if distance <= 8.0:
        return "second_shell"
    if distance <= 12.0:
        return "pocket_periphery"
    return "distal"


def choose_distance_reference(row: pd.Series) -> tuple[float, str, str]:
    source_groups = [
        (
            [
                "dist_to_experimental_ligand_NAM_median",
                "dist_to_experimental_ligand_PRPP_median",
            ],
            "experimental_ligand",
            "high",
        ),
        (
            [
                "dist_to_transferred_ligand_NAM_median",
                "dist_to_transferred_ligand_PRPP_median",
            ],
            "transferred_ligand",
            "medium",
        ),
        (
            ["dist_to_catalytic_residues_median"],
            "catalytic_residues",
            "medium",
        ),
    ]
    for columns, source_type, confidence in source_groups:
        values = [
            float(row[column]) for column in columns
            if column in row and pd.notna(row[column])
        ]
        if values:
            return min(values), source_type, confidence
    return float("nan"), "none", "low"


def build_structure_features(
    sequence: str,
    protein_id: str,
    pdb_path: str | Path,
    catalytic_residues: set[int],
    chain_id: str = "A",
    sasa_backend: str = "freesasa_cli",
    structure_source: str | None = None,
    experimental_ligand_resnames: dict[str, set[str]] | None = None,
    transferred_ligand_atoms: dict[str, list[object]] | None = None,
    traceability_dir: str | Path | None = None,
    traceability_prefix: str | None = None,
) -> pd.DataFrame:
    sequence = validate_wt_sequence(sequence)
    residue_map, residue_mapping_statuses = align_structure_residues_to_wt(
        sequence, pdb_path, chain_id
    )
    sasa = calculate_residue_sasa(
        pdb_path, chain_id, backend=sasa_backend,
        artifact_dir=traceability_dir,
        artifact_prefix=(
            f"{traceability_prefix}_residue"
            if traceability_prefix else "residue"
        ),
    ).set_index("pdb_residue_key")
    interface = calculate_interface_features(
        pdb_path, chain_id, sasa_backend=sasa_backend,
        artifact_dir=traceability_dir,
        artifact_prefix=traceability_prefix,
    ).set_index("pdb_residue_key")
    dssp = calculate_dssp_features(
        pdb_path, chain_id,
        artifact_dir=traceability_dir,
        artifact_prefix=traceability_prefix,
    ).set_index("pdb_residue_key")
    catalytic_atoms = [
        atom
        for position in sorted(set(catalytic_residues) & set(residue_map))
        for atom in residue_map[position].get_atoms()
        if atom.element != "H"
    ]
    if experimental_ligand_resnames is None:
        experimental_ligand_resnames = {
            "NAM": {"NAM"}, "PRPP": {"PRP", "PRPP"}
        }
    transferred_ligand_atoms = (
        {} if transferred_ligand_atoms is None
        else transferred_ligand_atoms
    )
    experimental_nam_atoms = extract_named_ligand_atoms(
        pdb_path, experimental_ligand_resnames.get("NAM", set())
    )
    experimental_prpp_atoms = extract_named_ligand_atoms(
        pdb_path, experimental_ligand_resnames.get("PRPP", set())
    )
    all_atoms = [
        atom for residue in residue_map.values()
        for atom in residue.get_atoms() if atom.element != "H"
    ]
    neighbor_search = NeighborSearch(all_atoms)
    backbone_angles = calculate_backbone_angles(residue_map)
    sequence_hash = hashlib.sha256(sequence.encode("ascii")).hexdigest()
    if structure_source is None:
        stem = Path(pdb_path).stem.lower()
        structure_source = (
            "synthetic_smoke_fixture"
            if "fake" in stem or "synthetic" in stem
            else "user_supplied_wt_structure"
        )
    rows = []
    for position, wt_aa in enumerate(sequence, start=1):
        if position not in residue_map:
            rows.append({
                "protein_id": protein_id,
                "sequence_hash": sequence_hash,
                "position": position,
                "wt_aa": wt_aa,
                "structure_source": structure_source,
                "structure_model_count": 1,
                "structure_chain_id": chain_id,
                "pdb_residue_number": np.nan,
                "pdb_insertion_code": "",
                "residue_mapping_status": residue_mapping_statuses.get(
                    position, "missing_in_structure"
                ),
                "plddt_or_structure_confidence": np.nan,
                "active_site_shell": "unknown",
                "secondary_structure": "unknown",
                "secondary_structure_consensus": "unknown",
                "dimer_interface": False,
                "dimer_interface_consensus": False,
                "interface_partner_present": False,
                "partner_heavy_atom_contact": False,
                "feature_confidence": "unavailable",
                "row_status": "failed",
                "error_message": (
                    "PDB residue is missing or identity-mismatched at "
                    f"WT position {position}"
                ),
            })
            continue
        residue = residue_map[position]
        pdb_number, pdb_insertion_code, pdb_key = residue_identifier(
            residue
        )
        if pdb_key not in sasa.index:
            raise ValueError(f"SASA row missing for PDB residue {pdb_key}")
        sasa_row = sasa.loc[pdb_key]
        nearby_residues = {
            nearby
            for nearby in neighbor_search.search(
                residue["CA"].coord, 8.0, level="R"
            )
            if nearby is not residue and is_aa(nearby, standard=True)
        }
        catalytic_distance = minimum_heavy_atom_distance(
            residue, catalytic_atoms
        )
        experimental_nam_distance = minimum_heavy_atom_distance(
            residue, experimental_nam_atoms
        )
        experimental_prpp_distance = minimum_heavy_atom_distance(
            residue, experimental_prpp_atoms
        )
        transferred_nam_distance = minimum_heavy_atom_distance(
            residue, transferred_ligand_atoms.get("NAM", [])
        )
        transferred_prpp_distance = minimum_heavy_atom_distance(
            residue, transferred_ligand_atoms.get("PRPP", [])
        )
        relative_sasa = float(sasa_row["relative_sasa"])
        contact_count = len(nearby_residues)
        phi, psi = backbone_angles[position]
        interface_row = interface.loc[pdb_key]
        dssp_row = dssp.loc[pdb_key]
        if pd.notna(dssp_row["phi_dssp"]):
            phi = float(dssp_row["phi_dssp"])
        if pd.notna(dssp_row["psi_dssp"]):
            psi = float(dssp_row["psi_dssp"])
        interface_delta = float(interface_row["interface_delta_sasa"])
        partner_contact = bool(interface_row["partner_heavy_atom_contact"])
        partner_present = bool(interface_row["interface_partner_present"])
        rows.append({
            "protein_id": protein_id,
            "sequence_hash": sequence_hash,
            "position": position,
            "wt_aa": wt_aa,
            "structure_source": structure_source,
            "structure_model_count": 1,
            "structure_chain_id": chain_id,
            "pdb_residue_number": pdb_number,
            "pdb_insertion_code": pdb_insertion_code,
            "residue_mapping_status": residue_mapping_statuses[position],
            "plddt_or_structure_confidence": (
                float(residue["CA"].get_bfactor())
                if structure_source
                and "colabfold" in str(structure_source).lower()
                and residue.has_id("CA")
                else np.nan
            ),
            "relative_sasa_median": relative_sasa,
            "relative_sasa_iqr": 0.0,
            "total_sasa_median": float(sasa_row["total_sasa"]),
            "total_sasa_iqr": 0.0,
            "sidechain_sasa_median": float(sasa_row["sidechain_sasa"]),
            "sidechain_sasa_iqr": 0.0,
            "sasa_backend": str(sasa_row["sasa_backend"]),
            "burial_class": (
                "buried" if relative_sasa < 5.0
                else "partially_exposed" if relative_sasa < 25.0
                else "exposed"
            ),
            "contact_count_median": contact_count,
            "contact_count_iqr": 0.0,
            "packing_class": (
                "loosely_packed" if contact_count <= 4
                else "moderately_packed" if contact_count <= 8
                else "densely_packed"
            ),
            "phi_median": phi,
            "phi_iqr": 0.0,
            "psi_median": psi,
            "psi_iqr": 0.0,
            "backbone_angle_unit": "degrees",
            "dist_to_experimental_ligand_NAM_median": experimental_nam_distance,
            "dist_to_experimental_ligand_NAM_iqr": 0.0,
            "dist_to_experimental_ligand_PRPP_median": experimental_prpp_distance,
            "dist_to_experimental_ligand_PRPP_iqr": 0.0,
            "dist_to_transferred_ligand_NAM_median": transferred_nam_distance,
            "dist_to_transferred_ligand_NAM_iqr": 0.0,
            "dist_to_transferred_ligand_PRPP_median": transferred_prpp_distance,
            "dist_to_transferred_ligand_PRPP_iqr": 0.0,
            "dist_to_catalytic_residues_median": catalytic_distance,
            "dist_to_catalytic_residues_iqr": 0.0,
            "distance_reference_type": "pending",
            "distance_reference_confidence": "pending",
            "dist_to_selected_reference_median": np.nan,
            "active_site_shell": "pending",
            "secondary_structure": str(
                dssp_row["secondary_structure_dssp"]
            ),
            "secondary_structure_consensus": str(
                dssp_row["secondary_structure_dssp"]
            ),
            "dssp_status": str(dssp_row["dssp_status"]),
            "dimer_interface": bool(
                partner_present and (interface_delta > 1.0 or partner_contact)
            ),
            "dimer_interface_consensus": bool(
                partner_present and (interface_delta > 1.0 or partner_contact)
            ),
            "interface_partner_present": partner_present,
            "partner_heavy_atom_contact": partner_contact,
            "interface_delta_sasa_median": interface_delta,
            "interface_delta_sasa_iqr": 0.0,
            "feature_confidence": (
                "smoke_only"
                if structure_source == "synthetic_smoke_fixture"
                else "single_structure"
            ),
            "row_status": "success",
            "error_message": "",
        })
    result = pd.DataFrame(rows)
    selected = result.apply(choose_distance_reference, axis=1)
    result["dist_to_selected_reference_median"] = [x[0] for x in selected]
    result["distance_reference_type"] = [x[1] for x in selected]
    result["distance_reference_confidence"] = [
        "test_fixture" if structure_source == "synthetic_smoke_fixture" else x[2]
        for x in selected
    ]
    result["active_site_shell"] = result[
        "dist_to_selected_reference_median"
    ].map(assign_structure_bin)
    if len(result) != len(sequence):
        raise AssertionError("Structure output lost WT positions")
    if "ligand_distance" in result.columns:
        raise AssertionError("Ambiguous ligand_distance column is forbidden")
    return result


def _finite_median_iqr(values) -> tuple[float, float]:
    numeric = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if numeric.empty:
        return float("nan"), float("nan")
    median = float(numeric.median())
    iqr = float(numeric.quantile(0.75) - numeric.quantile(0.25))
    return median, iqr


def _deterministic_mode(values):
    series = pd.Series(values).dropna()
    if series.empty:
        return "unknown", 0.0
    counts = series.astype(str).value_counts()
    maximum = int(counts.max())
    winners = sorted(counts[counts.eq(maximum)].index.tolist())
    return winners[0], maximum / len(series)


def aggregate_structure_ensemble(per_model: pd.DataFrame) -> pd.DataFrame:
    required = {
        "position", "protein_id", "sequence_hash", "wt_aa",
        "structure_model_id", "row_status",
    }
    missing = sorted(required - set(per_model.columns))
    if missing:
        raise ValueError(f"Per-model structure table is missing: {missing}")
    all_model_count = per_model["structure_model_id"].nunique()
    successful = per_model.loc[per_model["row_status"].eq("success")].copy()
    if successful.empty:
        raise ValueError("No successful WT structure models are available")
    numeric_bases = [
        "relative_sasa", "total_sasa", "sidechain_sasa",
        "contact_count", "phi", "psi",
        "dist_to_experimental_ligand_NAM",
        "dist_to_experimental_ligand_PRPP",
        "dist_to_transferred_ligand_NAM",
        "dist_to_transferred_ligand_PRPP",
        "dist_to_catalytic_residues", "interface_delta_sasa",
        "dist_to_selected_reference",
        "ligand_transfer_aligned_ca_count",
        "ligand_transfer_aligned_coverage",
        "ligand_transfer_rmsd_angstrom",
    ]
    categorical_columns = [
        "burial_class", "packing_class", "active_site_shell",
        "secondary_structure", "dimer_interface",
        "interface_partner_present", "partner_heavy_atom_contact",
        "dssp_status", "ligand_transfer_accepted",
    ]
    rows = []
    for position, group in successful.groupby("position", sort=True):
        first = group.iloc[0]
        row = {
            "protein_id": first["protein_id"],
            "sequence_hash": first["sequence_hash"],
            "position": int(position),
            "wt_aa": first["wt_aa"],
            "structure_source": "wt_structure_ensemble",
            "structure_model_ids": ";".join(
                sorted(group["structure_model_id"].astype(str).unique())
            ),
            "successful_structure_models": group[
                "structure_model_id"
            ].nunique(),
            "structure_model_count": group[
                "structure_model_id"
            ].nunique(),
            "total_structure_models": all_model_count,
            "sasa_backend": ";".join(
                sorted(group["sasa_backend"].astype(str).unique())
            ),
            "row_status": "success",
            "error_message": "",
        }
        failed_rows = per_model.loc[
            per_model["position"].eq(position)
            & per_model["row_status"].ne("success")
        ]
        row["failed_structure_models"] = failed_rows[
            "structure_model_id"
        ].nunique()
        row["ensemble_warning"] = "; ".join(
            sorted(
                value for value in failed_rows.get(
                    "error_message", pd.Series(dtype=str)
                ).dropna().astype(str).unique()
                if value
            )
        )
        mapping_mode, _ = _deterministic_mode(
            group.get(
                "residue_mapping_status",
                pd.Series("aligned_identity", index=group.index),
            )
        )
        row["residue_mapping_status"] = mapping_mode
        confidence_median, confidence_iqr = _finite_median_iqr(
            group.get(
                "plddt_or_structure_confidence",
                pd.Series(np.nan, index=group.index),
            )
        )
        row["plddt_or_structure_confidence"] = confidence_median
        row["plddt_or_structure_confidence_iqr"] = confidence_iqr
        for base in numeric_bases:
            source = f"{base}_median"
            if source not in group.columns:
                continue
            median, iqr = _finite_median_iqr(group[source])
            row[source] = median
            row[f"{base}_iqr"] = iqr
        for column in categorical_columns:
            if column not in group.columns:
                continue
            mode, agreement = _deterministic_mode(group[column])
            row[column] = (
                mode == "True"
                if column in {
                    "dimer_interface", "interface_partner_present",
                    "partner_heavy_atom_contact", "ligand_transfer_accepted",
                }
                else mode
            )
            row[f"{column}_agreement"] = agreement
        row["backbone_angle_unit"] = "degrees"
        rows.append(row)
    successful_positions = set(successful["position"].astype(int))
    all_positions = sorted(set(per_model["position"].astype(int)))
    for position in all_positions:
        if position in successful_positions:
            continue
        group = per_model.loc[per_model["position"].eq(position)]
        first = group.iloc[0]
        rows.append({
            "protein_id": first["protein_id"],
            "sequence_hash": first["sequence_hash"],
            "position": int(position),
            "wt_aa": first["wt_aa"],
            "structure_source": "wt_structure_ensemble",
            "structure_model_ids": ";".join(
                sorted(group["structure_model_id"].astype(str).unique())
            ),
            "successful_structure_models": 0,
            "structure_model_count": 0,
            "total_structure_models": all_model_count,
            "failed_structure_models": group[
                "structure_model_id"
            ].nunique(),
            "ensemble_warning": "; ".join(
                sorted(
                    value for value in group.get(
                        "error_message", pd.Series(dtype=str)
                    ).dropna().astype(str).unique()
                    if value
                )
            ),
            "residue_mapping_status": "missing_in_all_structures",
            "plddt_or_structure_confidence": np.nan,
            "plddt_or_structure_confidence_iqr": np.nan,
            "secondary_structure": "unknown",
            "dimer_interface": False,
            "active_site_shell": "unknown",
            "row_status": "failed",
            "error_message": "No successful structure mapping for this WT position",
        })
    result = pd.DataFrame(rows)
    selected = result.apply(choose_distance_reference, axis=1)
    result["dist_to_selected_reference_median"] = [x[0] for x in selected]
    result["distance_reference_type"] = [x[1] for x in selected]
    result["distance_reference_confidence"] = [x[2] for x in selected]
    result["active_site_shell"] = result[
        "dist_to_selected_reference_median"
    ].map(assign_structure_bin)
    result["secondary_structure_consensus"] = result[
        "secondary_structure"
    ]
    result["dimer_interface_consensus"] = result["dimer_interface"]
    dssp_success_counts = successful.assign(
        _dssp_success=successful.get(
            "dssp_status", pd.Series("unknown", index=successful.index)
        ).eq("success")
    ).groupby("position")["_dssp_success"].sum()
    result["dssp_successful_structure_models"] = (
        result["position"].map(dssp_success_counts).fillna(0).astype(int)
    )
    result["dssp_success_fraction"] = (
        result["dssp_successful_structure_models"]
        / result["successful_structure_models"].replace(0, np.nan)
    ).fillna(0.0)
    result["feature_confidence"] = np.select(
        [
            result["successful_structure_models"].ge(3)
            & result["failed_structure_models"].eq(0)
            & result["dssp_success_fraction"].ge(0.95),
            result["successful_structure_models"].ge(2)
            & result["dssp_success_fraction"].ge(0.50),
        ],
        ["high", "medium"],
        default="low",
    )
    result.loc[
        result["row_status"].ne("success"), "feature_confidence"
    ] = "unavailable"
    return result.sort_values("position").reset_index(drop=True)


In [6]:

if TEST_MODE:
    WT50 = SMOKE_WT_SEQUENCE
    smoke_output_dir = Path(SMOKE_OUTPUT_DIR)
    smoke_output_dir.mkdir(parents=True, exist_ok=True)
    fake_pdb_path = smoke_output_dir / "fake50.pdb"
    write_synthetic_pdb(WT50, fake_pdb_path)
    structure_frame = build_structure_features(
        WT50,
        "FAKE50",
        fake_pdb_path,
        catalytic_residues={10, 20},
        sasa_backend=SMOKE_SASA_BACKEND,
        structure_source="synthetic_smoke_fixture",
    )
    assert len(structure_frame) == 50
    assert structure_frame["position"].tolist() == list(range(1, 51))
    smoke_path = smoke_output_dir / "wt_structure_features_smoke.csv"
    structure_frame.to_csv(smoke_path, index=False)
    print(
        f"Smoke output: {smoke_path} ({len(structure_frame)} rows, "
        f"SASA backend={SMOKE_SASA_BACKEND})"
    )
else:
    print("Production mode: 50-aa smoke cell skipped.")


Production mode: 50-aa smoke cell skipped.


In [7]:

import hashlib
import json
import shutil
import subprocess
import urllib.request


def _log_tail(text, max_chars=4000):
    text = text or ""
    return text[-max_chars:]


def _resolve_configured_input(filename):
    if not filename:
        return None
    if INPUT_MODE == "upload":
        path = UPLOADED_PATHS.get(Path(filename).name)
        if path is None:
            raise FileNotFoundError(f"Requested upload was not provided: {filename}")
        return Path(path)
    candidate = Path(filename)
    return candidate if candidate.is_absolute() else INPUT_DIR / candidate


def _read_fasta(path):
    lines = [
        line.strip() for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.startswith(">")
    ]
    return validate_wt_sequence("".join(lines))


def _download_if_missing(url, path, minimum_bytes=1000):
    """Cache a small public reference file in Drive using an atomic rename."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size >= minimum_bytes:
        print(f"Reusing cached reference: {path}")
        return path
    partial = path.with_suffix(path.suffix + ".part")
    try:
        with urllib.request.urlopen(url, timeout=120) as response:
            payload = response.read()
        if len(payload) < minimum_bytes:
            raise RuntimeError(
                f"Downloaded reference is unexpectedly small ({len(payload)} bytes): {url}"
            )
        partial.write_bytes(payload)
        partial.replace(path)
    finally:
        if partial.exists():
            partial.unlink()
    print(f"Downloaded reference: {path}")
    return path


def _configure_catalytic_mapping(sequence, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    mapping_path = output_dir / "catalytic_residue_mapping.csv"
    summary_path = output_dir / "catalytic_residue_mapping_summary.json"
    if MANUAL_CATALYTIC_RESIDUES:
        invalid = sorted(
            position for position in MANUAL_CATALYTIC_RESIDUES
            if int(position) < 1 or int(position) > len(sequence)
        )
        if invalid:
            raise ValueError(f"Manual catalytic positions are outside the WT sequence: {invalid}")
        manual = sorted({int(position) for position in MANUAL_CATALYTIC_RESIDUES})
        frame = pd.DataFrame({
            "target_position": manual,
            "target_aa": [sequence[position - 1] for position in manual],
            "role": "manually_curated",
            "mapping_status": "manual",
            "overall_mapping_accepted": True,
        })
        summary = {
            "source": "manual", "mapping_accepted": True,
            "accepted_target_positions": manual,
        }
        selected = set(manual)
    elif AUTO_MAP_CATALYTIC_RESIDUES:
        reference_dir = output_dir / "reference_assets"
        # Human NAMPT canonical sequence: UniProt P43490.fasta
        reference_fasta = _download_if_missing(
            HUMAN_NAMPT_FASTA_URL,
            reference_dir / "P43490.fasta",
            minimum_bytes=400,
        )
        reference_sequence = _read_fasta(reference_fasta)
        frame, summary = map_reference_active_site_residues(
            reference_sequence,
            sequence,
            REFERENCE_ACTIVE_SITE_DEFINITIONS,
            minimum_identity=MIN_REFERENCE_IDENTITY,
            minimum_reference_coverage=MIN_REFERENCE_COVERAGE,
        )
        selected = accepted_mapped_positions(frame)
        summary.update({
            "source": "UniProt_P43490_sequence_alignment",
            "reference_fasta": str(reference_fasta),
            "accepted_target_positions": sorted(selected),
        })
        if not selected:
            frame.to_csv(mapping_path, index=False)
            summary_path.write_text(
                json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            raise RuntimeError(
                "No curated human NAMPT anchor could be mapped safely. Inspect "
                f"{mapping_path}, then provide MANUAL_CATALYTIC_RESIDUES if appropriate."
            )
    else:
        frame = pd.DataFrame(columns=[
            "target_position", "target_aa", "role", "mapping_status",
            "overall_mapping_accepted",
        ])
        summary = {
            "source": "disabled", "mapping_accepted": False,
            "accepted_target_positions": [],
        }
        selected = set()
    frame.to_csv(mapping_path, index=False)
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Catalytic-site mapping:", summary)
    return frame, summary, selected


STRUCTURE_SUFFIXES = {".pdb", ".cif", ".mmcif"}


def _cached_colabfold_pdbs(output_dir, state, num_models):
    recorded = [Path(path) for path in state.get("pdb_files", [])]
    existing = [path for path in recorded if path.exists()]
    if not existing:
        patterns = ["*_unrelaxed_rank_*.pdb", "ranked_*.pdb", "*.pdb"]
        for pattern in patterns:
            existing = sorted(Path(output_dir).glob(pattern))
            if existing:
                break
    return existing[:num_models]


def predict_wt_dimer_with_colabfold(
    sequence, protein_id, output_dir, num_models=3, num_recycles=3
):
    sequence = validate_wt_sequence(sequence)
    sequence_sha256 = hashlib.sha256(sequence.encode("ascii")).hexdigest()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fasta_path = output_dir / f"{protein_id}_homodimer.fasta"
    state_path = output_dir / "colabfold_state.json"
    expected_fasta = f">{protein_id}_homodimer\n{sequence}:{sequence}\n"

    if state_path.exists():
        try:
            state = json.loads(state_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            state = {}
        legacy_sequence_match = (
            fasta_path.exists()
            and fasta_path.read_text(encoding="utf-8") == expected_fasta
        )
        sequence_matches = (
            state.get("input_sequence_sha256") == sequence_sha256
            or legacy_sequence_match
        )
        cached_paths = _cached_colabfold_pdbs(output_dir, state, num_models)
        if (
            state.get("status") == "complete"
            and sequence_matches
            and len(cached_paths) >= num_models
        ):
            print(f"Reusing cached ColabFold WT models: {cached_paths}")
            return cached_paths

    # Only require the executable/GPU when a new prediction is actually needed.
    if shutil.which("colabfold_batch") is None:
        raise RuntimeError(
            "colabfold_batch is unavailable. Keep RUN_COLABFOLD=True, rerun the "
            "install cell, then manually restart the runtime if Colab requests it."
        )
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("A new ColabFold structure prediction requires a GPU runtime")
    fasta_path.write_text(expected_fasta, encoding="utf-8")
    command = [
        "colabfold_batch", str(fasta_path), str(output_dir),
        "--model-type", "alphafold2_multimer_v3",
        "--num-models", str(num_models),
        "--num-recycle", str(num_recycles),
    ]
    state = {
        "status": "started",
        "command": command,
        "input_sequence_sha256": sequence_sha256,
        "num_models_requested": int(num_models),
        "num_recycles": int(num_recycles),
        "amber_relaxation": False,
        "wt_only": True,
    }
    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
    completed = subprocess.run(command, text=True, capture_output=True)
    stdout_path = output_dir / "colabfold_stdout.log"
    stderr_path = output_dir / "colabfold_stderr.log"
    stdout_path.write_text(completed.stdout or "", encoding="utf-8")
    stderr_path.write_text(completed.stderr or "", encoding="utf-8")
    if completed.returncode != 0:
        stderr_tail = _log_tail(completed.stderr)
        stdout_tail = _log_tail(completed.stdout)
        print("===== ColabFold stdout tail =====\n" + stdout_tail)
        print("===== ColabFold stderr tail =====\n" + stderr_tail)
        status = "oom" if "out of memory" in (completed.stderr or "").lower() else "failed"
        state.update({
            "status": status, "returncode": completed.returncode,
            "stdout_log": str(stdout_path), "stderr_log": str(stderr_path),
            "stderr_tail": stderr_tail,
        })
        state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
        if status == "oom":
            raise RuntimeError(
                "CUDA OOM. State/logs were saved. Manually restart a clean runtime "
                "and reduce model count or MSA depth; automatic reconnect is disabled."
            )
        raise RuntimeError(
            f"ColabFold failed (exit {completed.returncode}). Full logs: {stderr_path}"
        )
    pdb_paths = _cached_colabfold_pdbs(output_dir, {}, num_models)
    if len(pdb_paths) < num_models:
        raise FileNotFoundError(
            f"ColabFold produced {len(pdb_paths)} PDB files; expected {num_models}."
        )
    state.update({"status": "complete", "pdb_files": [str(p) for p in pdb_paths]})
    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
    return pdb_paths


REFERENCE_COMPLEX_PATH = None
CATALYTIC_MAPPING_FRAME = pd.DataFrame()
CATALYTIC_MAPPING_SUMMARY = {}
if TEST_MODE:
    CATALYTIC_RESIDUES = set()
    STRUCTURE_PATHS = [fake_pdb_path]
    STRUCTURE_ORIGINS = {Path(fake_pdb_path): "synthetic_smoke_fixture"}
    print("Smoke mode: synthetic monomer validates mapping and SASA plumbing only.")
else:
    if FASTA_FILENAME:
        WT_SEQUENCE = _read_fasta(_resolve_configured_input(FASTA_FILENAME))
    else:
        WT_SEQUENCE = validate_wt_sequence(WT_SEQUENCE)
    print("Validated WT sequence length:", len(WT_SEQUENCE))

    CATALYTIC_MAPPING_FRAME, CATALYTIC_MAPPING_SUMMARY, CATALYTIC_RESIDUES = (
        _configure_catalytic_mapping(WT_SEQUENCE, MODULE2_DIR)
    )

    if REFERENCE_COMPLEX_FILENAME:
        if AUTO_DOWNLOAD_REFERENCE_COMPLEX:
            # Reference complex: RCSB 3DKL.pdb (PRP plus benzamide analogue).
            REFERENCE_COMPLEX_PATH = _download_if_missing(
                REFERENCE_COMPLEX_URL,
                MODULE2_DIR / "reference_assets" / "3DKL.pdb",
                minimum_bytes=10_000,
            )
        else:
            REFERENCE_COMPLEX_PATH = _resolve_configured_input(
                REFERENCE_COMPLEX_FILENAME
            )
            if not REFERENCE_COMPLEX_PATH.exists():
                raise FileNotFoundError(
                    f"Reference complex not found: {REFERENCE_COMPLEX_PATH}"
                )

    configured = [
        _resolve_configured_input(name) for name in WT_STRUCTURE_FILENAMES
    ]
    if INPUT_MODE == "upload" and not configured:
        configured = [
            Path(path) for path in UPLOADED_PATHS.values()
            if Path(path).suffix.lower() in STRUCTURE_SUFFIXES
            and Path(path).name != Path(REFERENCE_COMPLEX_FILENAME).name
        ]
    if INPUT_MODE == "drive" and not configured:
        configured = sorted(
            path for path in INPUT_DIR.iterdir()
            if path.is_file()
            and path.suffix.lower() in STRUCTURE_SUFFIXES
            and path.name != Path(REFERENCE_COMPLEX_FILENAME).name
        )
    missing = [str(path) for path in configured if not path.exists()]
    if missing:
        raise FileNotFoundError("Configured WT structures not found: " + "; ".join(missing))
    if configured:
        STRUCTURE_PATHS = configured
        STRUCTURE_ORIGINS = {
            Path(path): "user_supplied_wt_structure" for path in STRUCTURE_PATHS
        }
        print(f"Using {len(STRUCTURE_PATHS)} WT structure(s):", STRUCTURE_PATHS)
    elif RUN_COLABFOLD:
        STRUCTURE_PATHS = predict_wt_dimer_with_colabfold(
            WT_SEQUENCE, PROTEIN_ID, MODULE2_DIR / "colabfold_wt_dimer",
            num_models=COLABFOLD_NUM_MODELS,
            num_recycles=COLABFOLD_NUM_RECYCLES,
        )
        STRUCTURE_ORIGINS = {
            Path(path): "colabfold_wt_dimer" for path in STRUCTURE_PATHS
        }
    else:
        raise RuntimeError(
            "No WT PDB/mmCIF found. Put structure files in Drive inputs/, list them in "
            "WT_STRUCTURE_FILENAMES, use INPUT_MODE='upload', or set RUN_COLABFOLD=True."
        )


Validated WT sequence length: 467
Reusing cached reference: /content/drive/MyDrive/nampt_zero_shot/module2/reference_assets/P43490.fasta
Catalytic-site mapping: {'aligned_pairs': 451, 'identical_pairs': 252, 'sequence_identity': 0.5587583148558758, 'reference_coverage': 0.9185336048879837, 'target_coverage': 0.9657387580299786, 'minimum_identity': 0.25, 'minimum_reference_coverage': 0.7, 'mapping_accepted': True, 'source': 'UniProt_P43490_sequence_alignment', 'reference_fasta': '/content/drive/MyDrive/nampt_zero_shot/module2/reference_assets/P43490.fasta', 'accepted_target_positions': [203, 240, 304]}
Reusing cached reference: /content/drive/MyDrive/nampt_zero_shot/module2/reference_assets/3DKL.pdb
Reusing cached ColabFold WT models: [PosixPath('/content/drive/MyDrive/nampt_zero_shot/module2/colabfold_wt_dimer/FAKE50_homodimer_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'), PosixPath('/content/drive/MyDrive/nampt_zero_shot/module2/colabfold_wt_dimer/FAKE50_homodimer_u

In [8]:

if TEST_MODE:
    print("Smoke mode already calculated positional features.")
else:
    import hashlib
    import importlib.metadata
    import traceback

    module2_dir = MODULE2_DIR
    module2_dir.mkdir(parents=True, exist_ok=True)
    STRUCTURE_LOG_DIR.mkdir(parents=True, exist_ok=True)
    per_model_frames = []
    failure_rows = []
    for model_index, structure_path in enumerate(STRUCTURE_PATHS, start=1):
        structure_path = Path(structure_path)
        model_id = f"wt_model_{model_index}"
        structure_origin = STRUCTURE_ORIGINS.get(
            Path(structure_path), "user_supplied_wt_structure"
        )
        try:
            transfer_result = {
                "accepted": False, "aligned_ca_count": 0,
                "aligned_coverage": 0.0, "rmsd_angstrom": np.nan,
                "nam_atoms": [], "prpp_atoms": [],
                "failure_reason": "reference complex not configured",
            }
            if REFERENCE_COMPLEX_PATH is not None:
                transfer_result = transfer_reference_ligands(
                    structure_path,
                    REFERENCE_COMPLEX_PATH,
                    target_chain_id=TARGET_CHAIN_ID,
                    reference_chain_id=REFERENCE_CHAIN_ID,
                )
            transferred_atoms = (
                {
                    "NAM": transfer_result["nam_atoms"],
                    "PRPP": transfer_result["prpp_atoms"],
                }
                if transfer_result["accepted"] else {}
            )
            frame = build_structure_features(
                WT_SEQUENCE,
                PROTEIN_ID,
                structure_path,
                catalytic_residues=set(CATALYTIC_RESIDUES),
                chain_id=TARGET_CHAIN_ID,
                sasa_backend="freesasa_cli",
                structure_source=structure_origin,
                transferred_ligand_atoms=transferred_atoms,
                traceability_dir=STRUCTURE_LOG_DIR,
                traceability_prefix=model_id,
            )
            frame["structure_model_id"] = model_id
            frame["structure_file"] = str(structure_path)
            frame["structure_file_sha256"] = hashlib.sha256(
                structure_path.read_bytes()
            ).hexdigest()
            frame["reference_complex_sha256"] = (
                hashlib.sha256(REFERENCE_COMPLEX_PATH.read_bytes()).hexdigest()
                if REFERENCE_COMPLEX_PATH is not None else ""
            )
            frame["ligand_transfer_accepted"] = transfer_result["accepted"]
            frame["ligand_transfer_aligned_ca_count"] = transfer_result[
                "aligned_ca_count"
            ]
            frame["ligand_transfer_aligned_coverage"] = transfer_result[
                "aligned_coverage"
            ]
            frame["ligand_transfer_rmsd_angstrom"] = transfer_result[
                "rmsd_angstrom"
            ]
            frame["ligand_transfer_failure_reason"] = transfer_result[
                "failure_reason"
            ]
            if transfer_result["accepted"]:
                coordinate_rows = []
                for substrate, atoms in [
                    ("NAM", transfer_result["nam_atoms"]),
                    ("PRPP", transfer_result["prpp_atoms"]),
                ]:
                    for atom_index, atom in enumerate(atoms, start=1):
                        coordinate_rows.append({
                            "structure_model_id": model_id,
                            "substrate": substrate,
                            "atom_index": atom_index,
                            "atom_name": atom.get_name(),
                            "x": float(atom.coord[0]),
                            "y": float(atom.coord[1]),
                            "z": float(atom.coord[2]),
                        })
                pd.DataFrame(coordinate_rows).to_csv(
                    module2_dir / f"{model_id}_transferred_ligand_coordinates.csv",
                    index=False,
                )
            per_model_frames.append(frame)
        except Exception as exc:
            details = traceback.format_exc()
            print(f"===== {model_id} structure failure =====\n" + details)
            (STRUCTURE_LOG_DIR / f"{model_id}_structure_failure.log").write_text(
                details, encoding="utf-8"
            )
            for position, wt_aa in enumerate(
                validate_wt_sequence(WT_SEQUENCE), start=1
            ):
                failure_rows.append({
                    "protein_id": PROTEIN_ID,
                    "sequence_hash": hashlib.sha256(
                        validate_wt_sequence(WT_SEQUENCE).encode("ascii")
                    ).hexdigest(),
                    "position": position,
                    "wt_aa": wt_aa,
                    "structure_model_id": model_id,
                    "structure_file": str(structure_path),
                    "structure_source": structure_origin,
                    "row_status": "failed",
                    "error_message": f"{type(exc).__name__}: {exc}",
                })
    if not per_model_frames:
        failure_path = module2_dir / "wt_structure_failures.csv"
        pd.DataFrame(failure_rows).to_csv(failure_path, index=False)
        raise RuntimeError(
            f"All WT structures failed; details saved to {failure_path}"
        )
    per_model = pd.concat(
        per_model_frames + ([pd.DataFrame(failure_rows)] if failure_rows else []),
        ignore_index=True, sort=False,
    )
    per_model_path = module2_dir / "wt_structure_features_per_model.csv"
    per_model.to_csv(per_model_path, index=False)
    module2_frame = aggregate_structure_ensemble(per_model)
    module2_frame["mapped_catalytic_residues"] = ";".join(
        str(position) for position in sorted(CATALYTIC_RESIDUES)
    )
    module2_frame["catalytic_mapping_source"] = CATALYTIC_MAPPING_SUMMARY.get(
        "source", "unavailable"
    )
    module2_frame["catalytic_reference_accession"] = (
        "UniProt:P43490" if AUTO_MAP_CATALYTIC_RESIDUES else "manual_or_disabled"
    )
    module2_frame["catalytic_mapping_identity"] = CATALYTIC_MAPPING_SUMMARY.get(
        "sequence_identity", np.nan
    )
    module2_frame["catalytic_mapping_reference_coverage"] = (
        CATALYTIC_MAPPING_SUMMARY.get("reference_coverage", np.nan)
    )
    module2_frame["catalytic_mapping_accepted"] = bool(
        CATALYTIC_MAPPING_SUMMARY.get("mapping_accepted", False)
    )
    module2_frame["nam_smiles_hash"] = hashlib.sha256(
        NAM_SMILES.encode("utf-8")
    ).hexdigest()
    module2_frame["prpp_smiles_hash"] = hashlib.sha256(
        PRPP_SMILES.encode("utf-8")
    ).hexdigest()
    module2_frame["feature_scaling"] = "none_raw_values"
    versions = {}
    for package in ["pandas", "numpy", "biopython"]:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "unavailable"
    try:
        versions["freesasa_cli"] = subprocess.check_output(
            ["freesasa", "--version"], text=True, stderr=subprocess.STDOUT
        ).strip()
    except Exception:
        versions["freesasa_cli"] = "unavailable"
    used_colabfold = "colabfold_wt_dimer" in set(STRUCTURE_ORIGINS.values())
    if used_colabfold:
        try:
            versions["colabfold"] = importlib.metadata.version("colabfold")
        except importlib.metadata.PackageNotFoundError:
            versions["colabfold"] = "unavailable"
    module2_frame["package_versions"] = json.dumps(
        versions, ensure_ascii=False, sort_keys=True
    )
    module2_frame["colabfold_num_models_requested"] = (
        COLABFOLD_NUM_MODELS if used_colabfold else 0
    )
    module2_frame["colabfold_num_recycles"] = (
        COLABFOLD_NUM_RECYCLES if used_colabfold else 0
    )
    module2_frame["colabfold_amber_relaxation"] = False
    structure_frame = module2_frame
    print(f"Saved per-model raw features: {per_model_path}")
    print(f"Mapped catalytic positions: {sorted(CATALYTIC_RESIDUES)}")
    print(f"Saved FreeSASA/DSSP raw artifacts and logs: {STRUCTURE_LOG_DIR}")


Saved per-model raw features: /content/drive/MyDrive/nampt_zero_shot/module2/wt_structure_features_per_model.csv
Mapped catalytic positions: [203, 240, 304]
Saved FreeSASA/DSSP raw artifacts and logs: /content/drive/MyDrive/nampt_zero_shot/module2/logs


In [9]:

output_frame = structure_frame if TEST_MODE else module2_frame
if output_frame is None:
    raise RuntimeError("No Notebook 2 output frame is available")

expected_rows = len(validate_wt_sequence(WT_SEQUENCE))
if len(output_frame) != expected_rows:
    raise AssertionError(
        f"Expected {expected_rows} WT-position rows, found {len(output_frame)}"
    )
if output_frame["position"].nunique() != expected_rows:
    raise AssertionError("WT position rows are duplicated or missing")
if "ligand_distance" in output_frame.columns:
    raise AssertionError("Ambiguous ligand_distance column is forbidden")

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
filename = (
    "wt_structure_features_smoke.csv" if TEST_MODE
    else "wt_structure_features.csv"
)
local_path = LOCAL_OUTPUT_DIR / filename
output_frame.to_csv(local_path, index=False)
print(f"Saved local CSV: {local_path}")

if DRIVE_MOUNTED:
    MODULE2_DIR.mkdir(parents=True, exist_ok=True)
    drive_path = MODULE2_DIR / filename
    output_frame.to_csv(drive_path, index=False)
    print(f"Saved Drive CSV: {drive_path}")

print("Rows:", len(output_frame))
print("Row status:")
print(output_frame["row_status"].value_counts(dropna=False))
if "successful_structure_models" in output_frame.columns:
    print("Successful structure models:")
    print(output_frame["successful_structure_models"].value_counts(dropna=False))
if "feature_confidence" in output_frame.columns:
    print("Feature confidence:")
    print(output_frame["feature_confidence"].value_counts(dropna=False))

preview_columns = [
    "position", "wt_aa", "relative_sasa_median", "relative_sasa_iqr",
    "contact_count_median", "secondary_structure", "dimer_interface",
    "dist_to_selected_reference_median", "distance_reference_type",
    "feature_confidence",
]
display(output_frame[[
    column for column in preview_columns if column in output_frame.columns
]].head(10))


Saved local CSV: /content/nampt_zero_shot/wt_structure_features.csv
Saved Drive CSV: /content/drive/MyDrive/nampt_zero_shot/module2/wt_structure_features.csv
Rows: 467
Row status:
row_status
success    467
Name: count, dtype: int64
Successful structure models:
successful_structure_models
3    467
Name: count, dtype: int64
Feature confidence:
feature_confidence
high    467
Name: count, dtype: int64


,position,wt_aa,relative_sasa_median,relative_sasa_iqr,contact_count_median,secondary_structure,dimer_interface,dist_to_selected_reference_median,distance_reference_type,feature_confidence
0,1,M,104.1,3.85,4.0,coil,True,25.938627,transferred_ligand,high
1,2,Q,78.9,16.10,7.0,coil,False,25.178205,transferred_ligand,high
2,3,P,33.1,2.40,12.0,coil,False,21.788523,transferred_ligand,high
3,4,N,10.4,1.20,15.0,coil,False,19.847263,transferred_ligand,high
4,5,I,1.4,0.25,18.0,G,False,17.463976,transferred_ligand,high
5,6,I,0.0,0.00,19.0,G,False,17.994036,transferred_ligand,high
6,7,L,3.3,0.25,16.0,G,True,16.778904,transferred_ligand,high
7,8,L,7.8,0.50,14.0,S,True,13.553180,transferred_ligand,high
8,9,T,0.0,0.00,17.0,S,True,11.683384,transferred_ligand,high
9,10,D,1.1,0.25,15.0,B,True,6.934855,transferred_ligand,high
